In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [7]:
from getpass import getpass
import duckdb
import os

os.environ["HF_TOKEN"] = getpass("Paste your Hugging Face read token: ")

con = duckdb.connect()
con.sql(f"CREATE SECRET hf_token (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}');")

Paste your Hugging Face read token: ··········


┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [8]:
REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [9]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()
clients.head(10)

,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


In [11]:
end_d = con.sql(f"SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}").df()["end_d"][0]
print("Latest date in data:", end_d)

features = con.sql(f"""
    WITH windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  DATE '{end_d}' - INTERVAL 90 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last90,
               SUM(CASE WHEN f.report_date <= DATE '{end_d}' - INTERVAL 90 DAY
                        AND f.report_date >  DATE '{end_d}' - INTERVAL 180 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev90,
               SUM(CASE WHEN f.report_date >  DATE '{end_d}' - INTERVAL 90 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last90,
               SUM(CASE WHEN f.report_date <= DATE '{end_d}' - INTERVAL 90 DAY
                        AND f.report_date >  DATE '{end_d}' - INTERVAL 180 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_prev90,
               AVG(CASE WHEN f.report_date >  DATE '{end_d}' - INTERVAL 90 DAY THEN f.gsc_avg_position END)       AS pos_last90,
               STDDEV(CASE WHEN f.report_date >  DATE '{end_d}' - INTERVAL 90 DAY THEN f.gsc_avg_position END)    AS pos_volatility_90d
        FROM {TABLES['fact_daily']} f
        WHERE f.report_date > DATE '{end_d}' - INTERVAL 180 DAY
        GROUP BY 1, 2
        HAVING imp_prev90 >= 200
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Latest date in data: 2026-06-30 00:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

104,725 content items with enough history


,client_hash_id,content_hash_id,imp_last90,imp_prev90,clk_last90,clk_prev90,pos_last90,pos_volatility_90d
0,client_e547b89c05043229,content_dab4cb3e1444bb8e,1576.0,2353.0,4.0,4.0,19.966225,13.351111
1,client_e547b89c05043229,content_4a586e6347428a1d,17141.0,17852.0,21.0,21.0,7.904428,1.418361
2,client_e547b89c05043229,content_93271239b3d644f6,457.0,502.0,5.0,0.0,13.323680,14.777718
3,client_e547b89c05043229,content_794699b07967595f,952.0,868.0,1.0,0.0,29.744296,22.235612
4,client_e547b89c05043229,content_a6f62e0eccfc64e3,257.0,511.0,0.0,1.0,17.248655,18.518549


In [12]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']

data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows, {data["top_query_share"].notna().sum():,} with query signals')
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 104,725 rows, 84,979 with query signals


,client_hash_id,content_hash_id,imp_last90,imp_prev90,clk_last90,clk_prev90,pos_last90,pos_volatility_90d,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_dab4cb3e1444bb8e,1576.0,2353.0,4.0,4.0,19.966225,13.351111,8.0,0.086929,0.663071,164.0,394.0,0.416244
1,client_e547b89c05043229,content_4a586e6347428a1d,17141.0,17852.0,21.0,21.0,7.904428,1.418361,38.0,0.016685,0.847325,487.0,2331.0,0.208923
2,client_e547b89c05043229,content_93271239b3d644f6,457.0,502.0,5.0,0.0,13.323680,14.777718,1.0,0.153173,0.816193,14.0,14.0,1.000000
3,client_e547b89c05043229,content_794699b07967595f,952.0,868.0,1.0,0.0,29.744296,22.235612,4.0,0.302521,0.628151,25.0,66.0,0.378788
4,client_e547b89c05043229,content_a6f62e0eccfc64e3,257.0,511.0,0.0,1.0,17.248655,18.518549,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
data['is_declining'] = (
    (data['imp_last90'] < 0.8 * data['imp_prev90']) &
    (data['clk_last90'] <= data['clk_prev90'])
).astype(int)
print(data['is_declining'].value_counts(normalize=True))

is_declining
0    0.543376
1    0.456624
Name: proportion, dtype: float64


In [14]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, precision_score

# --- Baseline: transparent hand-rule score ---
def baseline_score(row):
    score = 0
    if row['imp_prev90'] > 0 and row['imp_last90'] < 0.8 * row['imp_prev90']: score += 2
    if row['pos_last90'] > 15: score += 1
    if row['pos_volatility_90d'] > 10: score += 1
    return score

data['baseline_score'] = data.apply(baseline_score, axis=1)

# --- Features for the model (leakage-safe: no imp_last90/clk_last90, those define the label) ---
FEATURES = ['imp_prev90', 'clk_prev90', 'pos_last90', 'pos_volatility_90d',
            'visible_queries', 'rare_share', 'anon_share', 'top_query_share']
model_data = data.dropna(subset=FEATURES + ['is_declining']).copy()
X = model_data[FEATURES]
y = model_data['is_declining']

# --- Split by client, not by row ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_data['client_hash_id']))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print("base rate (always predict majority):", round(y_test.value_counts(normalize=True).max(), 3))
print("\n--- MODEL ---")
print(classification_report(y_test, preds, digits=3))

# --- Baseline comparison on same test set ---
test_baseline = model_data.iloc[test_idx]['baseline_score']
baseline_preds = (test_baseline >= 2).astype(int)
print("--- BASELINE (hand-rule) ---")
print(classification_report(y_test, baseline_preds, digits=3))

# Feature importance — what's actually driving the model
import pandas as pd
importances = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nFeature importance:\n", importances)

base rate (always predict majority): 0.534

--- MODEL ---
              precision    recall  f1-score   support

           0      0.720     0.868     0.787     11235
           1      0.802     0.613     0.695      9796

    accuracy                          0.749     21031
   macro avg      0.761     0.741     0.741     21031
weighted avg      0.758     0.749     0.744     21031

--- BASELINE (hand-rule) ---
              precision    recall  f1-score   support

           0      1.000     0.513     0.678     11235
           1      0.642     1.000     0.782      9796

    accuracy                          0.740     21031
   macro avg      0.821     0.757     0.730     21031
weighted avg      0.833     0.740     0.726     21031


Feature importance:
 imp_prev90            0.232938
pos_volatility_90d    0.137006
rare_share            0.127626
visible_queries       0.126811
anon_share            0.113115
pos_last90            0.094205
top_query_share       0.086372
clk_prev90          